# Ders 5A — Class Imbalance, Oversampling ve Undersampling

Bu notebookta pozitif/negatif oranlarını değiştirip RF performansını ve metrikleri karşılaştıracağız.

## 1. Paket kurulumu

Bu hücre Colab ortamında eksik paketleri kurar. RDKit moleküler fingerprint üretmek için gereklidir.

In [ ]:
# Google Colab için gerekli paketleri kuruyoruz.
# RDKit moleküllerden fingerprint üretmek için gerekir.
# SHAP/LIME yalnızca ilgili notebookta kullanılacak; burada temel paketleri kontrol ediyoruz.

import sys, subprocess, importlib.util

def pip_install(package_name):
    print(f"[INSTALL] {package_name} kuruluyor...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

for pkg, pip_name in [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("joblib", "joblib"),
]:
    if importlib.util.find_spec(pkg) is None:
        pip_install(pip_name)

# RDKit Colab ortamında bazen rdkit, bazen rdkit-pypi ile kuruluyor.
if importlib.util.find_spec("rdkit") is None:
    try:
        pip_install("rdkit")
    except Exception:
        pip_install("rdkit-pypi")

print("✅ Paket kontrolü tamamlandı.")

## 2. Paketleri çağırma

Bu hücrede analiz boyunca kullanacağımız paketleri import ediyoruz.

In [ ]:
# Bu hücrede kullanacağımız Python paketlerini çağırıyoruz.
# pandas: tablo/veri işlemleri
# numpy: sayısal işlemler
# matplotlib: grafik çizimi
# scikit-learn: makine öğrenmesi modelleri ve metrikler
# RDKit: SMILES -> moleküler fingerprint dönüşümü

from pathlib import Path
import os
import json
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, average_precision_score,
    f1_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator, Descriptors
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator

print("✅ Importlar tamamlandı.")

## 3. Genel ayarlar

GitHub veri linkleri, target/SMILES kolonları ve çıktı klasörü burada tanımlanır.

In [ ]:
# ============================================================
# CONFIG
# ============================================================
# Bu iki veri seti GitHub üzerinden okunacak.
# GitHub'daki /blob/main/ linkleri raw linke otomatik çevriliyor.

DATA_URLS = {
    "A14_A17_ERa_BLA": "https://github.com/MOL-FEST/MOL_FEST_2026/blob/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv",
    "A15_A18_ERa_LUC_VM7": "https://github.com/MOL-FEST/MOL_FEST_2026/blob/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv",
}

# Ana analiz için default veri seti.
# İstersen bunu "A14_A17_ERa_BLA" yapabilirsin.
ACTIVE_DATASET_KEY = "A15_A18_ERa_LUC_VM7"

TARGET_COLUMN = "binary_label_agonist1_antagonist0"
SMILES_COLUMN = "QSAR-Ready SMILES"

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Morgan fingerprint ayarları.
MORGAN_BITS = 1024
MORGAN_RADIUS = 2

# Çıktılar bu klasöre kaydedilecek.
OUTPUT_ROOT = Path("molfest_outputs")
OUTPUT_ROOT.mkdir(exist_ok=True)

print("✅ Config hazır.")
print(f"Aktif veri seti: {ACTIVE_DATASET_KEY}")

## 4. Ortak yardımcı fonksiyonlar

Bu fonksiyonlar veri okuma, fingerprint üretme, model eğitme, metrik hesaplama ve çıktı kaydetme işlerini kolaylaştırır.

In [ ]:
# ============================================================
# ORTAK YARDIMCI FONKSİYONLAR
# ============================================================

def note(title, message=""):
    """Notebook çalışırken açıklayıcı bilgi yazdırmak için küçük yardımcı."""
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    if message:
        print(message)


def github_to_raw(url):
    """GitHub blob linkini raw CSV linkine çevirir."""
    if "github.com" in url and "/blob/" in url:
        url = url.replace("https://github.com/", "https://raw.githubusercontent.com/")
        url = url.replace("/blob/", "/")
    return url


def read_csv_flexible(url_or_path):
    """CSV ayıracını otomatik anlamaya çalışır: önce ';', sonra ','."""
    source = github_to_raw(str(url_or_path))
    last_error = None
    for sep in [";", ","]:
        try:
            df = pd.read_csv(source, sep=sep, encoding="utf-8-sig", low_memory=False)
            # Yanlış ayraçta genellikle tek kolon gelir.
            if df.shape[1] > 1:
                return df, sep, source
        except Exception as e:
            last_error = e
    raise RuntimeError(f"CSV okunamadı: {url_or_path}\nSon hata: {last_error}")


def detect_column(df, preferred, keywords):
    if preferred in df.columns:
        return preferred
    for c in df.columns:
        cl = c.lower()
        if any(k.lower() in cl for k in keywords):
            return c
    raise ValueError(f"Kolon bulunamadı. Beklenen: {preferred}. Mevcut kolonlar: {list(df.columns)}")


def load_dataset(dataset_key=ACTIVE_DATASET_KEY):
    """GitHub'dan veri setini okur, target/smiles kolonlarını kontrol eder."""
    if dataset_key not in DATA_URLS:
        raise ValueError(f"dataset_key şunlardan biri olmalı: {list(DATA_URLS.keys())}")

    df, sep, source = read_csv_flexible(DATA_URLS[dataset_key])
    target_col = detect_column(df, TARGET_COLUMN, ["binary_label", "label", "target", "class"])
    smiles_col = detect_column(df, SMILES_COLUMN, ["smiles"])

    note(
        "Veri okundu",
        f"Kaynak: {source}\n"
        f"Ayraç: {repr(sep)}\n"
        f"Satır sayısı: {df.shape[0]}\n"
        f"Kolon sayısı: {df.shape[1]}\n"
        f"Target kolonu: {target_col}\n"
        f"SMILES kolonu: {smiles_col}"
    )

    return df, target_col, smiles_col, sep, source


def prepare_target_and_smiles(df, target_col, smiles_col):
    """Target ve SMILES temizliği: eksik target ve eksik SMILES çıkarılır."""
    before = len(df)
    y_num = pd.to_numeric(df[target_col], errors="coerce")
    mask = y_num.notna() & df[smiles_col].notna()

    df2 = df.loc[mask].copy().reset_index(drop=True)
    y = pd.to_numeric(df2[target_col], errors="coerce").astype(int).to_numpy()

    classes = np.unique(y)
    if len(classes) != 2:
        raise ValueError(f"Binary classification bekleniyor; bulunan sınıflar: {classes}")

    # Eğer sınıflar 0/1 değilse küçük sınıfı 0, büyük sınıfı 1 yapıyoruz.
    if not set(classes).issubset({0, 1}):
        mapping = {classes[0]: 0, classes[1]: 1}
        y = np.array([mapping[v] for v in y], dtype=int)

    note(
        "Target ve SMILES temizliği",
        f"Önceki satır sayısı: {before}\n"
        f"Kullanılabilir satır sayısı: {len(df2)}\n"
        f"Çıkarılan satır sayısı: {before - len(df2)}\n"
        f"Sınıf dağılımı: {dict(pd.Series(y).value_counts().sort_index())}"
    )
    return df2, y


def plot_class_distribution(y, out_path=None, title="Sınıf dağılımı"):
    counts = pd.Series(y).value_counts().sort_index()
    plt.figure(figsize=(6, 4))
    plt.bar([str(i) for i in counts.index], counts.values)
    plt.xlabel("Sınıf")
    plt.ylabel("Molekül sayısı")
    plt.title(title)
    plt.tight_layout()
    if out_path is not None:
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        print(f"[Kaydedildi] {out_path}")
    plt.show()


def smiles_to_morgan(smiles, radius=MORGAN_RADIUS, n_bits=MORGAN_BITS):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    names = [f"Morgan_r{radius}_{i}" for i in range(n_bits)]
    rows, valid = [], []

    for smi in smiles:
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
        if mol is None:
            valid.append(False)
            continue
        fp = generator.GetFingerprint(mol)
        arr = np.zeros((n_bits,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(fp, arr)
        rows.append(arr)
        valid.append(True)

    return np.vstack(rows).astype(np.float32), names, np.array(valid, dtype=bool)


def smiles_to_maccs(smiles):
    names = [f"MACCS_{i}" for i in range(1, 167)]
    rows, valid = [], []

    for smi in smiles:
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
        if mol is None:
            valid.append(False)
            continue
        fp = MACCSkeys.GenMACCSKeys(mol)
        arr167 = np.zeros((167,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(fp, arr167)
        rows.append(arr167[1:])  # bit 0 gerçek MACCS key değildir
        valid.append(True)

    return np.vstack(rows).astype(np.float32), names, np.array(valid, dtype=bool)


def smiles_to_rdkit_descriptors(smiles):
    descriptor_names = [name for name, _ in Descriptors._descList]
    calc = MolecularDescriptorCalculator(descriptor_names)
    rows, valid = [], []

    for smi in smiles:
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
        if mol is None:
            valid.append(False)
            continue
        desc = np.array(calc.CalcDescriptors(mol), dtype=np.float32)
        desc = np.nan_to_num(desc, nan=0.0, posinf=0.0, neginf=0.0)
        rows.append(desc)
        valid.append(True)

    return np.vstack(rows).astype(np.float32), descriptor_names, np.array(valid, dtype=bool)


def build_feature_matrix(df, y, smiles_col, feature_set="morgan"):
    """SMILES'tan istenen feature setini üretir."""
    smiles = df[smiles_col].tolist()

    if feature_set == "morgan":
        X, names, valid = smiles_to_morgan(smiles)
    elif feature_set == "maccs":
        X, names, valid = smiles_to_maccs(smiles)
    elif feature_set == "rdkit_descriptors":
        X, names, valid = smiles_to_rdkit_descriptors(smiles)
    elif feature_set == "maccs_morgan":
        X1, n1, v1 = smiles_to_maccs(smiles)
        X2, n2, v2 = smiles_to_morgan(smiles)
        valid = v1 & v2
        X = np.hstack([X1, X2])
        names = n1 + n2
    elif feature_set == "morgan_rdkit":
        X1, n1, v1 = smiles_to_morgan(smiles)
        X2, n2, v2 = smiles_to_rdkit_descriptors(smiles)
        valid = v1 & v2
        X = np.hstack([X1, X2])
        names = n1 + n2
    elif feature_set == "maccs_morgan_rdkit":
        X1, n1, v1 = smiles_to_maccs(smiles)
        X2, n2, v2 = smiles_to_morgan(smiles)
        X3, n3, v3 = smiles_to_rdkit_descriptors(smiles)
        valid = v1 & v2 & v3
        X = np.hstack([X1, X2, X3])
        names = n1 + n2 + n3
    else:
        raise ValueError("feature_set bilinmiyor.")

    # Bu fonksiyonlarda tüm feature blokları aynı valid molekül sırasıyla üretildiği için
    # valid mask normalde aynı olur. Güvenlik için y ve df'yi maskeliyoruz.
    df_valid = df.loc[valid].reset_index(drop=True)
    y_valid = y[valid]

    note(
        "Feature matrisi üretildi",
        f"Feature set: {feature_set}\n"
        f"Geçerli molekül sayısı: {X.shape[0]}\n"
        f"Feature sayısı: {X.shape[1]}\n"
        f"İlk 5 feature: {names[:5]}"
    )

    return X, y_valid, names, df_valid


def make_rf(n_estimators=300, class_weight="balanced_subsample"):
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_features="sqrt",
        class_weight=class_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X).astype(float)


def metric_dict(y_true, y_pred, y_score):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "ROC": roc_auc_score(y_true, y_score),
        "AP": average_precision_score(y_true, y_score),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


def train_evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = get_scores(model, X_test)
    return model, y_pred, y_score, metric_dict(y_test, y_pred, y_score)


def save_metrics_predictions(outdir, name, df_test, smiles_col, y_test, y_pred, y_score, metrics):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    pred_df = pd.DataFrame({
        "SMILES": df_test[smiles_col].values if smiles_col in df_test.columns else np.arange(len(y_test)),
        "y_true": y_test,
        "y_pred": y_pred,
        "y_score_class1": y_score,
    })
    pred_df.to_csv(outdir / f"{name}_predictions.csv", index=False)
    pd.DataFrame([metrics]).to_csv(outdir / f"{name}_metrics.csv", index=False)

    print(f"[Kaydedildi] {outdir / f'{name}_predictions.csv'}")
    print(f"[Kaydedildi] {outdir / f'{name}_metrics.csv'}")


print("✅ Yardımcı fonksiyonlar hazır.")

## 5. Class imbalance ve sampling

Bu notebookta sınıf oranlarını değiştiriyoruz:
- pozitif/negatif dengeli
- pozitif 5 kat fazla
- negatif 5 kat fazla

Her senaryoda oversampling ve undersampling uyguluyoruz.  
Önemli: Sampling sadece train set üzerinde yapılır, test set asla değiştirilmez.

In [ ]:
lesson_out = OUTPUT_ROOT / "lesson09_resampling"
models_dir = lesson_out / "saved_models"
pred_dir = lesson_out / "predictions"
models_dir.mkdir(parents=True, exist_ok=True)
pred_dir.mkdir(parents=True, exist_ok=True)

df, target_col, smiles_col, sep, source = load_dataset(ACTIVE_DATASET_KEY)
df_clean, y = prepare_target_and_smiles(df, target_col, smiles_col)
X, y, feature_names, df_valid = build_feature_matrix(df_clean, y, smiles_col, feature_set="morgan")

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df_valid, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print("Train dağılımı:", dict(pd.Series(y_train).value_counts().sort_index()))
print("Test dağılımı:", dict(pd.Series(y_test).value_counts().sort_index()))

In [ ]:
def resample_to_ratio(y_train, target_pos_neg_ratio, method, random_state=42):
    rng = np.random.RandomState(random_state)
    pos_idx = np.where(y_train == 1)[0]
    neg_idx = np.where(y_train == 0)[0]
    n_pos, n_neg = len(pos_idx), len(neg_idx)
    r = float(target_pos_neg_ratio)

    if method == "oversampling":
        current = n_pos / n_neg
        if current < r:
            target_pos = int(np.ceil(r * n_neg))
            target_neg = n_neg
        else:
            target_pos = n_pos
            target_neg = int(np.ceil(n_pos / r))
        sampled_pos = rng.choice(pos_idx, size=target_pos, replace=(target_pos > n_pos))
        sampled_neg = rng.choice(neg_idx, size=target_neg, replace=(target_neg > n_neg))

    elif method == "undersampling":
        current = n_pos / n_neg
        if current < r:
            target_pos = n_pos
            target_neg = max(5, int(np.floor(n_pos / r)))
            target_neg = min(target_neg, n_neg)
        else:
            target_neg = n_neg
            target_pos = max(5, int(np.floor(r * n_neg)))
            target_pos = min(target_pos, n_pos)
        sampled_pos = rng.choice(pos_idx, size=target_pos, replace=False)
        sampled_neg = rng.choice(neg_idx, size=target_neg, replace=False)
    else:
        raise ValueError("method oversampling veya undersampling olmalı.")

    idx = np.concatenate([sampled_pos, sampled_neg])
    rng.shuffle(idx)
    return idx

scenarios = [
    ("balanced_1_to_1", 1.0),
    ("positive_5_to_1", 5.0),
    ("negative_5_to_1", 0.2),
]
methods = ["oversampling", "undersampling"]

In [ ]:
rows = []
all_preds = []
count_rows = []

for scenario_name, ratio in scenarios:
    for method in methods:
        run_name = f"{scenario_name}_{method}"
        note(f"Sampling senaryosu: {run_name}", f"Hedef pozitif/negatif oranı: {ratio}")

        idx_res = resample_to_ratio(y_train, ratio, method, RANDOM_STATE)
        X_res = X_train[idx_res]
        y_res = y_train[idx_res]

        counts_after = dict(pd.Series(y_res).value_counts().sort_index())
        print("Sampling sonrası train dağılımı:", counts_after)

        rf = RandomForestClassifier(
            n_estimators=300,
            max_features="sqrt",
            class_weight=None,  # sampling etkisini izole etmek için kapalı
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf.fit(X_res, y_res)

        y_pred = rf.predict(X_test)
        y_score = get_scores(rf, X_test)
        metrics = metric_dict(y_test, y_pred, y_score)
        metrics.update({
            "Scenario": scenario_name,
            "Method": method,
            "TargetPosNegRatio": ratio,
            "TrainAfter_Positive": int(np.sum(y_res == 1)),
            "TrainAfter_Negative": int(np.sum(y_res == 0)),
        })
        rows.append(metrics)

        pred_df = pd.DataFrame({
            "Scenario": scenario_name,
            "Method": method,
            "SMILES": df_test[smiles_col].values,
            "y_true": y_test,
            "y_pred": y_pred,
            "y_score_class1": y_score,
        })
        pred_df.to_csv(pred_dir / f"{run_name}_predictions.csv", index=False)
        all_preds.append(pred_df)

        joblib.dump({"model": rf, "feature_names": feature_names, "scenario": run_name}, models_dir / f"{run_name}_rf.joblib")

        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["class 0", "class 1"])
        plt.title(f"Confusion Matrix — {run_name}")
        plt.tight_layout()
        plt.savefig(lesson_out / f"confusion_matrix_{run_name}.png", dpi=300, bbox_inches="tight")
        plt.show()

        print(f"✅ {run_name}: ROC={metrics['ROC']:.3f}, Recall={metrics['Recall']:.3f}, Specificity={metrics['Specificity']:.3f}, Precision={metrics['Precision']:.3f}")

result_df = pd.DataFrame(rows).sort_values("ROC", ascending=False)
result_df.to_csv(lesson_out / "resampling_metrics.csv", index=False)
pd.concat(all_preds, ignore_index=True).to_csv(lesson_out / "resampling_predictions_long.csv", index=False)

display(result_df)

In [ ]:
plt.figure(figsize=(12, 6))
labels = result_df["Scenario"] + "_" + result_df["Method"]
plt.barh(labels, result_df["BalancedAccuracy"])
plt.xlabel("Balanced Accuracy")
plt.title("Sampling senaryoları karşılaştırması")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(lesson_out / "resampling_balanced_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ Sampling dersi tamamlandı.")